# 🤖 Agentic Customer Service Chatbot
### BUS4-118S — Group Exercise: Agentic AI in Customer Service and Sales

**Data:** `data/` CSV + PDF files (no invented catalog)  
**Agents:** Order Status Agent | Refund Policy Agent | Product Suggestion Agent  
**Memory:** LangGraph MemorySaver (multi-turn conversation)  
**API:** Anthropic Claude Haiku 4.5 (`claude-haiku-4-5-20251001`)


## 1. Install Dependencies

In [ ]:
# Run this cell once to install required packages
%pip install langgraph langchain-core python-dotenv anthropic pandas pypdf --quiet


## 2. Load API Key Securely

Create a `.env` file in the same folder as this notebook with your [Anthropic](https://console.anthropic.com/) API key:

```
ANTHROPIC_API_KEY=your_key_here
```

Copy from `.env.example` if needed. `.env` is in `.gitignore` — never commit it to GitHub.


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
from anthropic import Anthropic
if not api_key:
    raise RuntimeError("Missing ANTHROPIC_API_KEY - check your .env file")
client = Anthropic(api_key=api_key)

print("✅ API Key loaded successfully!")


## 3. Load course data files

Reads **`data/Laptop Orders.csv`**, **`data/Laptop pricing.csv`**, and text extracted from **`EcoSprint_Specification_Document.pdf`** and **`Laptop product descriptions.pdf`** (there are no JSON files in this dataset). Run this before defining agents.


In [ ]:
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

DATA_DIR = Path("data")


def _pdf_to_text(filename: str, max_chars: int = 20000) -> str:
    path = DATA_DIR / filename
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages).strip()
    return text[:max_chars]


orders_df = pd.read_csv(DATA_DIR / "Laptop Orders.csv")
pricing_df = pd.read_csv(DATA_DIR / "Laptop pricing.csv")
ORDERS_TABLE = orders_df.to_string(index=False)
PRICING_TABLE = pricing_df.to_string(index=False)
SPEC_TEXT = _pdf_to_text("EcoSprint_Specification_Document.pdf")
PRODUCT_DESC_TEXT = _pdf_to_text("Laptop product descriptions.pdf")
print(
    f"Loaded {len(orders_df)} orders, {len(pricing_df)} price rows, spec PDF ({len(SPEC_TEXT)} chars), descriptions PDF ({len(PRODUCT_DESC_TEXT)} chars)."
)


## 4. Imports & LLM helper


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Literal
import operator

MODEL = "claude-haiku-4-5-20251001"


def _anthropic_text(response) -> str:
    parts = []
    for block in response.content:
        if hasattr(block, "text") and block.text:
            parts.append(block.text)
    return "".join(parts)


def invoke_claude(system_prompt: str, lc_messages: list) -> str:
    """Call Claude with a system prompt and LangChain Human/AI messages (no SystemMessage in list)."""
    msgs = []
    for m in lc_messages:
        if isinstance(m, HumanMessage):
            c = m.content if isinstance(m.content, str) else str(m.content)
            msgs.append({"role": "user", "content": c})
        elif isinstance(m, AIMessage):
            c = m.content if isinstance(m.content, str) else str(m.content)
            msgs.append({"role": "assistant", "content": c})
    resp = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        temperature=0,
        system=system_prompt.strip(),
        messages=msgs,
    )
    return _anthropic_text(resp)


print(f"✅ LLM ready: Anthropic {MODEL}")

## 5. Define State Schema


In [ ]:
# State tracks the conversation history across all turns
class ChatState(TypedDict):
    messages: Annotated[list, operator.add]  # Accumulates messages across turns
    intent: str                              # Which agent to route to
    user_name: str                           # Remembered from earlier in conversation

## 6. Define the Three Specialized Agents

Agents answer only from the tables and PDF text loaded in **§3** (no extra invented facts).


In [ ]:
# ─────────────────────────────────────────────
# AGENT 1: Order Status Agent
# ─────────────────────────────────────────────
ORDER_AGENT_SYSTEM_PROMPT = f"""
You are the Order Status Agent. Answer using ONLY the table below (from Laptop Orders.csv).
The only fields that exist are: Order ID, Product Ordered, Quantity Ordered, Delivery Date.
Do not invent tracking numbers, carriers, shipping status, or any other field.
If an Order ID is not in the table, say it is not in the data.

{ORDERS_TABLE}
"""


def order_status_agent(state: ChatState) -> ChatState:
    """Handles order status inquiries."""
    text = invoke_claude(ORDER_AGENT_SYSTEM_PROMPT, state["messages"])
    return {"messages": [AIMessage(content=f"[Order Status Agent] {text}")]}


# ─────────────────────────────────────────────
# AGENT 2: Refund Policy Agent
# ─────────────────────────────────────────────
REFUND_AGENT_SYSTEM_PROMPT = f"""
You are the Refund & Returns Agent. Answer ONLY from the document text below (EcoSprint_Specification_Document.pdf).
Do not invent return windows, fees, phone numbers, or policies that are not stated in the text.
If the text does not answer the question, say the document does not specify.

--- Document text ---
{SPEC_TEXT}
--- End document text ---
"""


def refund_policy_agent(state: ChatState) -> ChatState:
    """Handles refund policy and returns questions."""
    text = invoke_claude(REFUND_AGENT_SYSTEM_PROMPT, state["messages"])
    return {"messages": [AIMessage(content=f"[Refund Policy Agent] {text}")]}


# ─────────────────────────────────────────────
# AGENT 3: Product Suggestion Agent
# ─────────────────────────────────────────────
PRODUCT_AGENT_SYSTEM_PROMPT = f"""
You are the Product Recommendation Agent. Use ONLY the pricing table and description text below.
Do not mention laptop models, prices, shipping days, or specs that are not in this data.
If something is not in the data, say it is not in the catalog files.

Laptop pricing.csv:
{PRICING_TABLE}

Laptop product descriptions.pdf (extracted text):
{PRODUCT_DESC_TEXT}
"""


def product_suggestion_agent(state: ChatState) -> ChatState:
    """Handles product recommendations."""
    text = invoke_claude(PRODUCT_AGENT_SYSTEM_PROMPT, state["messages"])
    return {"messages": [AIMessage(content=f"[Product Suggestion Agent] {text}")]}


print("✅ Three agents defined: Order Status | Refund Policy | Product Suggestions")



## 7. Router Agent (Intent Detection)

The router reads the user's message and decides which specialized agent should handle it.


In [ ]:
ROUTER_SYSTEM_PROMPT = """
You are a routing assistant for customer service.
Read the customer's latest message and classify their intent into EXACTLY ONE of these categories:

- "order" — questions about order status, shipping, tracking, delivery
- "refund" — questions about returns, refunds, cancellations, billing issues
- "product" — questions about product recommendations, comparisons, what to buy, features, prices
- "general" — greetings, thanks, unclear requests, or anything else

Respond with ONLY the single word category (order, refund, product, or general). No other text.
"""

GENERAL_SYSTEM_PROMPT = """
You are a friendly customer service greeter.
Briefly welcome the customer and say you can help with order questions (from our order file), returns/policy questions (from our spec document), and product questions (from our pricing and product description files).
Do not invent company names, phone numbers, URLs, or policies.
Keep the reply short.
"""

def router_node(state: ChatState) -> ChatState:
    """Detects intent and routes to the correct agent."""
    # Only look at the last user message for routing
    last_user_msg = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            last_user_msg = msg.content
            break
    
    if not last_user_msg:
        return {"intent": "general"}
    
    routing_text = invoke_claude(ROUTER_SYSTEM_PROMPT, [HumanMessage(content=last_user_msg)])
    
    intent = routing_text.strip().lower()
    if intent not in ["order", "refund", "product", "general"]:
        intent = "general"  # fallback
    
    print(f"  🔀 Router → intent detected: '{intent}'")
    return {"intent": intent}

def general_agent(state: ChatState) -> ChatState:
    """Handles greetings and general questions."""
    text = invoke_claude(GENERAL_SYSTEM_PROMPT, state["messages"])
    return {"messages": [AIMessage(content=f"[Assistant] {text}")]}

def route_by_intent(state: ChatState) -> Literal["order_agent", "refund_agent", "product_agent", "general_agent"]:
    """Conditional edge — sends state to the right agent node."""
    intent_map = {
        "order": "order_agent",
        "refund": "refund_agent",
        "product": "product_agent",
        "general": "general_agent"
    }
    return intent_map.get(state["intent"], "general_agent")

print("✅ Router defined")

## 8. Build the LangGraph with MemorySaver


In [ ]:
# Initialize memory — persists state across conversation turns
memory = MemorySaver()

# Build the state graph
builder = StateGraph(ChatState)

# Add nodes
builder.add_node("router", router_node)
builder.add_node("order_agent", order_status_agent)
builder.add_node("refund_agent", refund_policy_agent)
builder.add_node("product_agent", product_suggestion_agent)
builder.add_node("general_agent", general_agent)

# Set entry point
builder.set_entry_point("router")

# Conditional routing from router to agents
builder.add_conditional_edges(
    "router",
    route_by_intent,
    {
        "order_agent": "order_agent",
        "refund_agent": "refund_agent",
        "product_agent": "product_agent",
        "general_agent": "general_agent"
    }
)

# All agents go to END after responding
builder.add_edge("order_agent", END)
builder.add_edge("refund_agent", END)
builder.add_edge("product_agent", END)
builder.add_edge("general_agent", END)

# Compile graph with memory
graph = builder.compile(checkpointer=memory)

print("✅ LangGraph compiled with MemorySaver")
print("\nGraph structure:")
print("  [START] → router → {order_agent | refund_agent | product_agent | general_agent} → [END]")

## 9. Chat Function


In [ ]:
def chat(user_input: str, thread_id: str = "session_1") -> str:
    """
    Send a message to the chatbot and get a response.
    thread_id identifies the conversation — same ID = same memory thread.
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    # Invoke graph with new user message
    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)], "intent": "", "user_name": ""},
        config=config
    )
    
    # Return the last AI message
    last_ai_msg = result["messages"][-1]
    return last_ai_msg.content


def chat_session(thread_id: str = "demo_session"):
    """
    Interactive chat session in the notebook.
    Type 'quit' or 'exit' to end the session.
    """
    print("="*60)
    print("  Welcome to customer service!")
    print("  (Type 'quit' to exit)")
    print("="*60)
    
    while True:
        user_input = input("\nYou: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("\nThank you for contacting us! Have a great day! 👋")
            break
        
        response = chat(user_input, thread_id=thread_id)
        print(f"\nBot: {response}")

print("✅ Chat functions ready")

## 10. Automated Demo — Shows All 3 Agent Types + Multi-Turn Memory

This demo runs scripted conversations to demonstrate each agent and memory across 6 turns (using real order IDs from the CSV).


In [ ]:
def run_demo():
    print("\n" + "="*65)
    print("  DEMO: Customer Service Chatbot")
    print("  Demonstrating: 3 Agent Types + Multi-Turn Memory")
    print("="*65)

    demo_thread = "demo_2026"

    conversation = [
        "Hi! I'm Alex and I just placed a few laptop orders on your site.",
        "Can you check order ORD-8276? I think I ordered a SpectraBook.",
        "What quantity and delivery date do you show for that same order?",
        "I also have order ORD-6948 for an OmegaPro G17. If I need to return it, what's the policy based on your docs?",
        "What if the laptop is defective — does the specification say anything about that?",
        "I'm comparing laptops under $2000 for programming. What do you recommend from your catalog?",
    ]

    for i, user_msg in enumerate(conversation, 1):
        print(f"\n{'─'*60}")
        print(f"Turn {i}")
        print(f"{'─'*60}")
        print(f"👤 User: {user_msg}")

        response = chat(user_msg, thread_id=demo_thread)
        print(f"\n🤖 Bot: {response}")

    print("\n" + "="*65)
    print("  ✅ DEMO COMPLETE — All 3 agents demonstrated, 6 turns of memory")
    print("="*65)

# Run the automated demo
run_demo()



## 11. Interactive Chat (Optional)

Run this cell to start a live interactive chat session in the notebook.


In [ ]:
# Uncomment the line below to start an interactive session
# chat_session(thread_id="my_session")

---
## Architecture Summary

```
User Input
    │
    ▼
[Router Agent]  ← reads intent from latest message
    │
    ├──► [Order Status Agent]    → order tracking, shipping info
    ├──► [Refund Policy Agent]   → returns, refunds, billing
    ├──► [Product Suggestion Agent] → recommendations, comparisons
    └──► [General Agent]         → greetings, fallback
    │
    ▼
[LangGraph MemorySaver]  ← persists full conversation history
    │
    ▼
Response to User
```

**Data:** `data/Laptop Orders.csv`, `data/Laptop pricing.csv`, plus PDF text from `EcoSprint_Specification_Document.pdf` and `Laptop product descriptions.pdf`.

**Memory:** `MemorySaver` stores the full message history per `thread_id`, so every agent has access to the complete prior conversation context.

**Routing:** The Router Agent uses Claude Haiku 4.5 to classify intent into one of 4 categories, then LangGraph's conditional edges dispatch to the right agent.


## 💬 Live Chat — Talk to the Chatbot

Run this cell to have a real conversation. Type 'quit' to exit.

In [ ]:
while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("Bot: Thanks for visiting! Goodbye! 👋")
        break
    response = chat(user_input, thread_id="live_session")
    print(f"Bot: {response}\n")